# # 과제명 : Multi Taget Classification 모델 생성

(분석 요건)
- 분석 주제 : KOSPI 지수 & 건강식품 지수의 상승 가능성(Probability) 예측
- 분석 단위 : Day
- 문제 유형 : 분류 문제
- 종속 변수 1 : D+3일 KOSPI 지수 상승 여부 (Target_1)
- 종속 변수 2 : D+3일 건강식품 지수 상승 여부 (Target_2)
- 설명 변수 : 외부 오픈데이터를 활용한 거시경제·금융시장·소비 트렌드·기상 요인
(해외 지수, 유가, 금리, 쇼핑 지수, 날씨 등)
- 학습 기간 : 924일 (약 2년 6.5개월)
- 평가 기간 : Public 평가 - 100일, Private 평가 - 200일

(평가 기준)
- 평가 지표 : AUC (Area Under the ROC Curve)
- 평가 Score : Public Score (전체 테스트 데이터 중 50%),  Private Score (전체 테스트 데이터 중 100%)
- 최종 평가 : 정량 평가 (Private Score) + 정성 평가 (보고서, Code)

# 데이터 준비
데이터 불러오기

In [ ]:
from pathlib import Path

import pandas as pd

# 프로젝트 루트 또는 notebooks/에서 실행할 수 있도록 경로 설정
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "submissions"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Train data
train = pd.read_csv(DATA_DIR / "train.csv", encoding="cp949")
train["Temp"] ="train"                                          # Dataset 구분 추가
train["Target_1"] = train["Target_1"].astype("int")             # Target_1 변수 타입 변경
train["Target_2"] = train["Target_2"].astype("int")             # Target_2 변수 타입 변경

# Test data
test = pd.read_csv(DATA_DIR / "test_public.csv", encoding="cp949")  # Public Test Set 불러오기
#test = pd.read_csv(DATA_DIR / "test_private.csv", encoding="cp949") # Private Test Set 불러오기
test["Temp"] ="test"                                            # Dataset 구분 추가



데이터 병합하기

In [ ]:
df = pd.concat([train, test], axis=0, ignore_index=True)
df = df.sort_values(by='Time_step', ascending=True).reset_index(drop=True)
df

# 파생변수 생성

In [ ]:
# === 전일 대비 증감율

import numpy as np

# 증감율 계산 대상 변수
cols = df.columns.difference(['Time_step', 'Target_1', 'Target_2', 'Weekday', 'Temp'])

# 증감율 변수 생성
df_rate = (df[cols] - df[cols].shift(1)) / df[cols].shift(1).replace(0, np.nan)

# 컬럼 이름 변경
df_rate = df_rate.add_suffix('_chg')

# 원본과 병합
df = pd.concat([df, df_rate], axis=1)

# 첫 번째 행은 전일값 없음으로 NaN → 제거
df = df.iloc[1:].reset_index(drop=True)

# Data Partition
 - 학습용과 평가용 데이타 우선 분리
 - 학습용 데이타는 Train vs Valid Set 으로 분리
 - !! Timeseries Split 주의 : Shuffle 사용 금지 (시간 순서 유지 필수)

In [ ]:
# Train 과 Test 분리
df_train = df.query("Temp == 'train'")
df_test  = df.query("Temp == 'test'")

# Train 과 Validation 분리
split_idx = int(len(df_train) * 0.7)

# 시계열 기준 분리
df_valid = df_train.iloc[split_idx:].reset_index(drop=True)
df_train = df_train.iloc[:split_idx].reset_index(drop=True)
print(df_train.shape)
print(df_valid.shape)
print(df_test.shape)

# 분석 변수 선택


원래 데이터 복제

In [ ]:
df_train_raw = df_train.copy()
df_valid_raw  = df_valid.copy()
df_test_raw  = df_test.copy()

분석 변수 선택

In [ ]:
df_train = df_train.drop(columns=["Time_step", "Temp"])
df_valid = df_valid.drop(columns=["Time_step", "Temp"])
df_test = df_test.drop(columns=["Time_step", "Temp"])

# X, Y 분리

In [ ]:
# Train
X_train = df_train.drop(columns=["Target_1", "Target_2"])
Y_train = df_train[['Target_1', 'Target_2']]

# Valid
X_valid = df_valid.drop(columns=["Target_1", "Target_2"])
Y_valid = df_valid[['Target_1', 'Target_2']]

# Test
X_test = df_test.drop(columns=["Target_1", "Target_2"])
Y_test = df_test[['Target_1', 'Target_2']]

# 단일값 변수 처리

In [ ]:
# 고유값 개수 계산 (NaN 포함)
unique_counts = X_train.nunique(dropna=False)

# 고유값이 1개 이하인 컬럼 식별
cols_to_drop = unique_counts[unique_counts <= 1].index.tolist()

# 데이터셋에서 삭제
X_train = X_train.drop(columns=cols_to_drop)
X_valid = X_valid.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop)

print(f"삭제된 컬럼: {cols_to_drop}")

# Missing 처리
Missing 데이터 확인

In [ ]:
df_train.isnull().sum()

Missing 대체 (연속형)
 - median 대체

In [ ]:
import numpy as np
import pandas as pd

# 연속형 변수 컬럼 자동 감지
continuous_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

# X_train에서 중앙값 계산
medians = X_train[continuous_cols].median()

# Missing 값을 중앙값으로 대체
X_train[continuous_cols] = X_train[continuous_cols].fillna(medians)
X_valid[continuous_cols] = X_valid[continuous_cols].fillna(medians)
X_test[continuous_cols] = X_test[continuous_cols].fillna(medians)

Missing 대체 (범주형)
 - New class ('Unknown') 대체

In [ ]:
# 신규 범주 (Unknown) 로 대체
import pandas as pd

# 범주형 컬럼 선택 (object + category)
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns

for col in cat_cols:
    # category 타입이면 Unknown 먼저 추가
    if X_train[col].dtype.name == 'category':
        X_train[col] = X_train[col].cat.add_categories('Unknown')
        X_valid[col] = X_valid[col].cat.add_categories('Unknown')
        X_test[col] = X_test[col].cat.add_categories('Unknown')

    # Missing → 'Unknown'
    X_train[col] = X_train[col].fillna('Unknown')
    X_valid[col] = X_valid[col].fillna('Unknown')
    X_test[col] = X_test[col].fillna('Unknown')

# 학습 준비

One-Hot Encoding

In [ ]:
import pandas as pd

# 범주형 컬럼 선택 (Train 기준)
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

# One-Hot Encoding
X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=False, dummy_na=False)
X_valid = pd.get_dummies(X_valid, columns=cat_cols, drop_first=False, dummy_na=False)
X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=False, dummy_na=False)

# Validation/Test를 Train의 컬럼 구조에 맞추기
# - Train에 있고 Validation/Test에 없는 컬럼 → 0으로 채움
# - Train에 없고 Validation/Test에 있는 컬럼 → 제거
X_valid = X_valid.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print("X_train shape:", X_train.shape)
print("X_valid shape:", X_valid.shape)
print("X_test  shape:", X_test.shape)

# RandomForest
 학습 -> 예측 -> 평가
  - API 문서 : https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Multi Target 처리 : Target별 학습
models = {}
for target in Y_train.columns:
    models[target] = RandomForestClassifier(
      n_estimators=100,
      max_depth=10,
      min_samples_split=10,
      min_samples_leaf=5,
      random_state=42
      )
    models[target].fit(X_train, Y_train[target])

# 예측 : '1의 확률' 추출
prob_df_valid = pd.DataFrame({target: models[target].predict_proba(X_valid)[:, 1] for target in Y_train.columns})  # Valid Set
prob_df_test = pd.DataFrame({target: models[target].predict_proba(X_test)[:, 1] for target in Y_train.columns})    # Test Set

# Valid set 평가
print("\n=== [Valid Set : Target별 AUC 점수] ===")
auc_list = []
for target in Y_train.columns:
    auc = roc_auc_score(Y_valid[target], prob_df_valid[target])
    auc_list.append(auc)
    print(f"{target:6s} ==> AUC: {auc:.4f}")
print(f"평균     ==> AUC: {np.mean(auc_list):.4f}")

- Test 결과 제출 파일 생성

In [ ]:
# Test Set 예측 결과
df_test_raw["pred_1"] = prob_df_test["Target_1"].values
df_test_raw["pred_2"] = prob_df_test["Target_2"].values
submission = df_test_raw[["Time_step", "pred_1", "pred_2"]]
submission.to_csv(OUTPUT_DIR / 'public_1반_박다솔_1.csv', sep=',', na_rep='NaN', index=False)    # Public Score 용, 홍길동_1: 제출자명_제출회차
#submission.to_csv(OUTPUT_DIR / 'private_1반_박다솔.csv', sep=',', na_rep='NaN', index=False)   # Private Score 용, 홍길동: 제출자명